kernel : Python (Pixi)

In [ ]:
import anndata
import gc
import glob
import os
import scanpy as sc
from anndata import AnnData
from scipy.sparse import csr_matrix
from pipeline.utils.env import find_env_dir

anndata.settings.allow_write_nullable_strings = True

raw_count_matrix_location = find_env_dir("RAW_COUNT_MATRIX")
h5ad_matrix_location = find_env_dir("H5AD_MATRIX")

def load_h5_data(data_dir: str, file_metadata: dict, series: str) -> AnnData:
    h5_files = glob.glob(os.path.join(data_dir, "*filtered_feature_bc_matrix.h5"))
    adata_list = []

    for file_path in h5_files:
        adata = sc.read_10x_h5(file_path, gex_only=True)

        file_name = os.path.basename(file_path)
        gsm_id = file_name.split("_")[0] 
        
        if gsm_id not in file_metadata:
            continue

        meta_info = file_metadata[gsm_id]
        adata.obs["sample"] = gsm_id
        adata.obs["stage"] = meta_info["stage"]
        adata.obs["condition"] = meta_info["condition"]
        
        adata.obs_names = adata.obs_names + "-" + gsm_id
        adata.var_names_make_unique()
        
        if not isinstance(adata.X, csr_matrix):
            adata.X = csr_matrix(adata.X)
            
        adata_list.append(adata)
    
    adata_concat = anndata.concat(adata_list, join="outer", merge="same")
    adata_concat.X = csr_matrix(adata_concat.X)
    adata_concat.obs["series"] = series

    return adata_concat

In [ ]:
FILE_METADATA = {
    "GSM7982308": {"stage": "Early", "condition": "EAE"},
    "GSM7982310": {"stage": "Early", "condition": "EAE"},
    "GSM7982312": {"stage": "Early", "condition": "EAE"},
    "GSM7982314": {"stage": "Early", "condition": "EAE"},
    "GSM7982316": {"stage": "Peak", "condition": "EAE"},
    "GSM7982318": {"stage": "Peak", "condition": "EAE"},
    "GSM7982320": {"stage": "Peak", "condition": "EAE"},
    "GSM7982322": {"stage": "Peak", "condition": "EAE"},
    "GSM7982324": {"stage": "Late", "condition": "EAE"},
    "GSM7982326": {"stage": "Late", "condition": "EAE"},
    "GSM7982328": {"stage": "Late", "condition": "EAE"},
    "GSM7982330": {"stage": "Late", "condition": "EAE"},
    "GSM7982332": {"stage": "Late", "condition": "EAE"},
    "GSM7982334": {"stage": "Peak", "condition": "Ctrl"},
    "GSM7982336": {"stage": "Early", "condition": "Ctrl"},
    "GSM7982338": {"stage": "Peak", "condition": "Ctrl"},
    "GSM7982340": {"stage": "Early", "condition": "Ctrl"},
    "GSM7982342": {"stage": "Late", "condition": "Ctrl"},
    "GSM7982344": {"stage": "Late", "condition": "Ctrl"},
    "GSM8637419": {"stage": "Early", "condition": "EAE"},
    "GSM8637421": {"stage": "Early", "condition": "EAE"},
    "GSM8637423": {"stage": "Early", "condition": "EAE"},
    "GSM8637425": {"stage": "Peak", "condition": "EAE"},
    "GSM8637427": {"stage": "Late", "condition": "EAE"},
    "GSM8637429": {"stage": "Naive", "condition": "Ctrl"},
    "GSM8637431": {"stage": "Early", "condition": "Ctrl"},
    "GSM8637433": {"stage": "Early", "condition": "Ctrl"},
    "GSM8637435": {"stage": "Early", "condition": "Ctrl"},
}
SERIES_NAME = "Zheng"
SAVE_FILE_NAME = "Zheng.h5ad"
adata = load_h5_data(os.path.join(raw_count_matrix_location, SERIES_NAME), file_metadata=FILE_METADATA, series = SERIES_NAME)

/home/sjkim/protocols/.pixi/envs/default/lib/python3.14/site-packages/anndata/_core/anndata.py:1798: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/home/sjkim/protocols/.pixi/envs/default/lib/python3.14/site-packages/anndata/_core/anndata.py:1798: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/home/sjkim/protocols/.pixi/envs/default/lib/python3.14/site-packages/anndata/_core/anndata.py:1798: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/home/sjkim/protocols/.pixi/envs/default/lib/python3.14/site-packages/anndata/_core/anndata.py:1798: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/home/sjkim/protocols/.pixi/envs/default/lib/python3.14/site-pac

,Xkr4,Gm1992,Gm19938,Gm37381,Rp1
AAACAGCCAGGAAGCC-1-GSM8637427,0.0,0.0,0.0,0.0,0.0
AAACAGCCAGGATTAA-1-GSM8637427,8.0,0.0,1.0,0.0,0.0
AAACATGCAACAGCCT-1-GSM8637427,7.0,0.0,0.0,0.0,0.0
AAACATGCAATCCTAG-1-GSM8637427,11.0,0.0,0.0,0.0,0.0
AAACATGCACTAAATC-1-GSM8637427,2.0,0.0,0.0,0.0,0.0


In [17]:
adata.var_names

Index(['Xkr4', 'Gm1992', 'Gm19938', 'Gm37381', 'Rp1', 'Sox17', 'Gm37587',
       'Gm37323', 'Mrpl15', 'Lypla1',
       ...
       'Gm16367', 'AC163611.1', 'AC163611.2', 'AC140365.1', 'AC124606.2',
       'AC124606.1', 'AC133095.2', 'AC133095.1', 'AC234645.1', 'AC149090.1'],
      dtype='str', length=32285)

In [18]:
adata.obs.index = adata.obs.index.astype(str)
adata.var.index = adata.var.index.astype(str)
adata.write_h5ad(os.path.join(h5ad_matrix_location, SAVE_FILE_NAME))
del adata
gc.collect()

892